# Refreshing the National Genomic Test Directory CodeSystems

This is the fourteenth notebook in the series - the first that writes its output
somewhere other than this repo. NW-GMSA's FHIR Implementation Guide
([nw-gmsa.github.io](https://nw-gmsa.github.io/en/), source
[github.com/nw-gmsa/nw-gmsa.github.com](https://github.com/nw-gmsa/nw-gmsa.github.com))
carries three `CodeSystem`s built from NHS England's published
[National Genomic Test Directories](https://www.england.nhs.uk/publication/national-genomic-test-directories/):

- **`GenomicTestCode`** (`EnglandTestCode.fsh`) - the legacy `R*`/`M*` test codes, one
  per specific laboratory investigation.
- **`GenomicClinicalIndication`** (`ClinicalIndication.fsh`) - the legacy 1st-level
  `R*`/`M*` clinical-indication grouping codes.
- **`DigitalGenomicTestServices`** (`DigitalGenomicTestServices.fsh`) - the *new*
  digital Test Directory coding, rolling out indication-by-indication: **TP** (Test
  Package) codes replacing `GenomicClinicalIndication`, **GT** (Genomic/Genetic Test)
  codes replacing `GenomicTestCode`'s `R*`/`M*`.

NHS England republishes each directory periodically (the Rare & Inherited Disease and
Cancer directories are already on versions 9 and 16), and more clinical areas migrate
to the new TP/GT scheme over time - this notebook is meant to be **rerun** each time
either happens, not run once. Three steps:

1. Check `GenomicTestCode`'s `R*`/`M*` codes against the current Rare & Inherited
   Disease (v9) and Cancer non-CNS (v16) master spreadsheets, and refresh it.
2. Check `DigitalGenomicTestServices` against the two directories that have already
   gone digital - Haematological Oncology (v1.1) and Cancer CNS (v2) - and refresh it.
3. Build a `ConceptMap` from the legacy codes those two directories' own spreadsheets
   say they replace, onto the new TP/GT codes.

Sources for steps 1-3 are all fetched live from the *public* `nw-gmsa/nw-gmsa.github.com`
repository and NHS England's own published spreadsheets - nothing vendored. The
refreshed FSH files are written into a local checkout of the IG's own repository,
`/Users/kevinmayfield/github/MFT/FHIRGenomics`, for review there (this notebook
doesn't commit anything - that's a deliberate separate step).

In [1]:
import os
import re
from collections import defaultdict

import openpyxl
import requests

FHIRGENOMICS_REPO = "/Users/kevinmayfield/github/MFT/FHIRGenomics"
FSH_CODESYSTEM_DIR = os.path.join(FHIRGENOMICS_REPO, "input/fsh/Termininology/CodeSystem")
FSH_CONCEPTMAP_DIR = os.path.join(FHIRGENOMICS_REPO, "input/fsh/Termininology/ConceptMap")
CACHE_DIR = "NotGit/NationalGenomicTestDirectory"  # gitignored - redownloaded fresh each run, never committed
os.makedirs(CACHE_DIR, exist_ok=True)

GITHUB_RAW = "https://raw.githubusercontent.com/nw-gmsa/nw-gmsa.github.com/main/input/fsh/Termininology/CodeSystem"


def fetch_current_fsh(filename):
    r = requests.get(f"{GITHUB_RAW}/{filename}")
    r.raise_for_status()
    return r.text


CODE_LINE_RE = re.compile(r'^\* #(\S+?),?\s+"(.*)"\s*$')  # tolerant of a stray comma after the code - see below


def parse_fsh_codes(text):
    codes = {}
    for line in text.splitlines():
        m = CODE_LINE_RE.match(line.strip())
        if m:
            codes[m.group(1)] = m.group(2)
    return codes


def parse_fsh_categories(text):
    """Map each concept's already-recorded `category` ^property value, if any.

    Used by DigitalGenomicTestServices.fsh's regeneration to carry `category`
    forward across a rerun rather than re-derive it (see that section's own
    markdown for why: some values there are curated by hand, not a clean
    function of the source spreadsheets). Only looks at direct `^property[+].code
    = #category` / `^property[=].valueCode = #x` pairs on a top-level concept -
    doesn't need to understand any other property (`parent`, `test-method`, ...)
    to do this.
    """
    categories = {}
    current_code = None
    awaiting_value = False
    for raw_line in text.splitlines():
        stripped = raw_line.strip()
        m = CODE_LINE_RE.match(stripped)
        if m and not raw_line.startswith(" "):
            current_code = m.group(1)
            awaiting_value = False
            continue
        if current_code is None:
            continue
        if not raw_line.startswith(" "):
            current_code = None
            awaiting_value = False
            continue
        if stripped == "* ^property[+].code = #category":
            awaiting_value = True
            continue
        if awaiting_value:
            vm = re.match(r'^\* \^property\[=\]\.valueCode = #(\S+)\s*$', stripped)
            if vm:
                categories[current_code] = vm.group(1)
            awaiting_value = False
    return categories


def natural_sort_key(code_str):
    return [int(p) if p.isdigit() else p for p in re.split(r"(\d+)", code_str)]


def fsh_escape(text):
    return text.replace("\\", "\\\\").replace('"', '\\"')


def build_concept_map_fsh(instance_name, title, description, source_url, source_version,
                           target_url, mapping, target_display, source_display=None, fixed_equivalence=None):
    lines = [
        f"Instance: {instance_name}",
        "InstanceOf: ConceptMap",
        f'Title: "{title}"',
        f'Description: """\n{description}\n"""',
        "Usage:  #definition",
        "",
        f'* name = "{instance_name}"',
        "* experimental = false",
        f'* url = "https://fhir.nwgenomics.nhs.uk/ConceptMap/{instance_name}"',
        f'* version = "{today}"',
        "* status = #active",
        "",
        f'* group.source = "{source_url}"',
        f'* group.sourceVersion = "{source_version}"',
        f'* group.target = "{target_url}"',
        f'* group.targetVersion = "{today}"',
        "",
    ]
    for source_code in sorted(mapping, key=natural_sort_key):
        targets = sorted(mapping[source_code])
        equivalence = fixed_equivalence or ("equivalent" if len(targets) == 1 else "relatedto")
        source_text = f' "{fsh_escape(source_display.get(source_code, ""))}"' if source_display else ""
        lines.append("* group.element[+]")
        lines.append(f"  * code = #{source_code}{source_text}")
        for target_code in targets:
            lines.append("  * target[+]")
            lines.append(f'    * code = #{target_code} "{fsh_escape(target_display.get(target_code, ""))}"')
            lines.append(f"    * equivalence = #{equivalence}")
    return "\n".join(lines) + "\n"

## The four NHS England master spreadsheets

In [2]:
SOURCES = {
    "rare-v9.xlsx": "https://www.england.nhs.uk/wp-content/uploads/2018/08/rare-and-inherited-disease-national-genomics-test-directory-v9.xlsx",
    "cancer-noncns-v16.xlsx": "https://www.england.nhs.uk/wp-content/uploads/2018/08/cancer-non-central-nervous-system-national-genomic-test-directory-version-16.xlsx",
    "haemonc-v1.1.xlsx": "https://www.england.nhs.uk/wp-content/uploads/2018/08/haematological-oncology-national-genomic-test-directory-version-1.1.xlsx",
    "cancer-cns-v2.xlsx": "https://www.england.nhs.uk/wp-content/uploads/2018/08/cancer-central-nervous-system-cns-national-genomic-test-directory-v2.xlsx",
}

xlsx_paths = {}
for filename, url in SOURCES.items():
    path = os.path.join(CACHE_DIR, filename)
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    with open(path, "wb") as f:
        f.write(r.content)
    xlsx_paths[filename] = path
    print(f"{filename:26} {len(r.content):>9,} bytes")

rare-v9.xlsx                 209,344 bytes
cancer-noncns-v16.xlsx       535,111 bytes


haemonc-v1.1.xlsx             65,517 bytes
cancer-cns-v2.xlsx            47,960 bytes


## Step 1: `GenomicTestCode` (`EnglandTestCode.fsh`)

Fetched live from the public repo - not this notebook's own working copy of
`FHIRGenomics`, which may be mid-edit locally.

In [3]:
current_fsh_text = fetch_current_fsh("EnglandTestCode.fsh")
current_codes = parse_fsh_codes(current_fsh_text)
current_r = {k: v for k, v in current_codes.items() if k.startswith("R")}
current_m = {k: v for k, v in current_codes.items() if k.startswith("M")}

comma_bug_lines = [l for l in current_fsh_text.splitlines() if re.match(r'^\* #\S+,\s', l.strip())]

print(f"{len(current_codes)} codes currently ({len(current_r)} R*, {len(current_m)} M*)")
print(f"{len(comma_bug_lines)} line(s) with a stray comma between the code and its display text - e.g.:")
for l in comma_bug_lines[:3]:
    print(" ", l)

2130 codes currently (457 R*, 1673 M*)
0 line(s) with a stray comma between the code and its display text - e.g.:


### Rare & Inherited Disease v9 → `R*`

One sheet, one row per test - `Test ID` is the `R*.*` code, display built as
`<Clinical Indication> (<Test Method>)`, matching the shape already used
throughout `EnglandTestCode.fsh` (e.g. `R14.1` = *"Acutely unwell children with a
likely monogenic disorder (WGS)"*).

Three `^property` values are recorded per code, not just the display text:

- **`category`** - fixed `#rare-and-inherited-disease` for every `R*` code.
- **`specialist-test-group`** - the spreadsheet's own "Specialist test group" column
  (`string`) - R* only, see the M* section below for why.
- **`test-method`** - the same "Test Method" column already used to build the display
  text above (`string`).

In [4]:
wb = openpyxl.load_workbook(xlsx_paths["rare-v9.xlsx"], data_only=True)
ws = wb["R&ID indications"]

r_master = {}
r_test_method = {}
r_specialist_group = {}
for row in range(2, ws.max_row + 1):
    test_id = ws.cell(row=row, column=2).value
    ci_name = ws.cell(row=row, column=3).value
    method = ws.cell(row=row, column=5).value
    specialist_group = ws.cell(row=row, column=7).value  # "Specialist test group"
    if test_id:
        test_id = test_id.strip()
        r_master[test_id] = f"{ci_name.strip()} ({method.strip()})"
        r_test_method[test_id] = method.strip()
        if specialist_group:
            r_specialist_group[test_id] = specialist_group.strip()

print(f"{len(r_master)} R* test codes in v9")
print(f"{len(r_test_method)} with a test-method value, {len(r_specialist_group)} with a specialist-test-group value")

457 R* test codes in v9
457 with a test-method value, 457 with a specialist-test-group value


In [5]:
r_added = sorted(set(r_master) - set(current_r), key=natural_sort_key)
r_removed = sorted(set(current_r) - set(r_master), key=natural_sort_key)
r_changed = sorted((k for k in set(r_master) & set(current_r) if r_master[k] != current_r[k]), key=natural_sort_key)

print(f"R*: {len(r_added)} new, {len(r_removed)} no longer in v9, {len(r_changed)} display text changed")
print()
print("New:", r_added)
print()
print("Sample display-text changes:")
for k in r_changed[:8]:
    print(f"  {k}")
    print(f"    was: {current_r[k]!r}")
    print(f"    now: {r_master[k]!r}")

R*: 0 new, 0 no longer in v9, 0 display text changed

New: []

Sample display-text changes:


### Cancer non-CNS v16 → `M*`

Three sheets (Solid tumours, Sarcoma, Paediatric), same column layout in each: `CI
code`/`Clinical indication name` are only populated on a group's first row (merged
cells in the spreadsheet itself) - forward-fill from the last seen value for the
rows below it. Display built as `<Clinical indication name>, <Test name>`.

`category` is fixed `#cancer` for every current `M*` code. `test-method` comes from
this sheet's own "Technology" column (the same concept the R* sheet calls "Test
Method") - skipped, not populated, when that cell is empty or holds multi-paragraph
guidance text instead of a short method name (two codes as of v16: `M4.14`, `M119.5`).
No `specialist-test-group` here - the Cancer directories don't carry an equivalent
column at all, unlike Rare & Inherited Disease.

In [6]:
wb = openpyxl.load_workbook(xlsx_paths["cancer-noncns-v16.xlsx"], data_only=True)

m_master = {}
m_test_method = {}
for sheet in ["Solid tumours", "Sarcoma", "Paediatric"]:
    ws = wb[sheet]
    ci_name = None
    for row in range(3, ws.max_row + 1):
        new_ci_name = ws.cell(row=row, column=4).value
        if new_ci_name:
            ci_name = new_ci_name
        test_code = ws.cell(row=row, column=5).value
        test_name = ws.cell(row=row, column=6).value
        technology = ws.cell(row=row, column=10).value  # "Technology" - test-method equivalent
        if test_code:
            test_code = test_code.strip()
            m_master[test_code] = f"{ci_name.strip()}, {test_name.strip()}"
            # A handful of Technology cells hold multi-paragraph guidance text instead
            # of a short method name (e.g. M4.14) - skip those rather than dumping
            # prose into a `test-method` property; others are just blank (M119.5).
            if technology and "\n" not in str(technology):
                m_test_method[test_code] = str(technology).strip()

print(f"{len(m_master)} M* test codes currently live across the three sheets in v16")
print(f"{len(m_master) - len(m_test_method)} of them have no usable Technology value for test-method")

493 M* test codes currently live across the three sheets in v16
2 of them have no usable Technology value for test-method


In [7]:
m_added = sorted(set(m_master) - set(current_m), key=natural_sort_key)
m_removed_from_v16 = sorted(set(current_m) - set(m_master), key=natural_sort_key)  # see legacy discussion below
m_changed = sorted((k for k in set(m_master) & set(current_m) if m_master[k] != current_m[k]), key=natural_sort_key)

print(f"M* in v16: {len(m_added)} new, {len(m_removed_from_v16)} not in v16, {len(m_changed)} display text changed")
print()
print("New:", m_added)
print()
print("Sample display-text changes:")
for k in m_changed[:8]:
    print(f"  {k}")
    print(f"    was: {current_m[k]!r}")
    print(f"    now: {m_master[k]!r}")

M* in v16: 0 new, 1180 not in v16, 0 display text changed

New: []

Sample display-text changes:


### The `M*` codes not in v16: legacy, not missing

`M*` codes minted for Haematological Oncology and Cancer CNS *before* those two
directories migrated to the digital TP/GT scheme (Step 2) no longer appear in any
current `M*`-coded master spreadsheet - v16 only ever covered Solid tumours/Sarcoma/
Paediatric. They're not being deleted (a `CodeSystem` shouldn't remove codes once
minted - old messages may still reference them), just no longer refreshable from a
live source. The new haem-onc/CNS spreadsheets do still name them, though, in their
own `Legacy 'M' codes` column - cross-checking against that confirms which of
`current_m`'s `v16`-absent codes are accounted-for legacy codes versus genuinely
unexplained.

Kept as two separate sets (not immediately unioned) specifically so each legacy code's
`category` can be derived from *which* spreadsheet actually named it -
`haematological-oncology` if the Haem Onc v1.1 sheet's own `Legacy 'M' codes` column
references it, `cancer` for everything else (referenced by the Cancer CNS v2 sheet's
own legacy column, or referenced by neither - the small "unexplained" set flagged
below already needs manual follow-up for other reasons, so `cancer` here is a stated
default for that handful, not a verified fact about each one individually). No
`test-method`/`specialist-test-group` for any legacy code - neither predecessor
directory's own spreadsheet is a source for those any more than the current one is.

In [8]:
def legacy_m_codes_referenced(xlsx_path, sheet, legacy_col_name):
    wb = openpyxl.load_workbook(xlsx_path, data_only=True)
    ws = wb[sheet]
    header = [ws.cell(row=2, column=c).value for c in range(1, ws.max_column + 1)]
    col = next(i + 1 for i, h in enumerate(header) if h and legacy_col_name in h)
    referenced = set()
    for row in range(3, ws.max_row + 1):
        value = ws.cell(row=row, column=col).value
        if value:
            referenced.update(c.strip() for c in str(value).split(",") if c.strip())
    return referenced


haemonc_legacy_referenced = legacy_m_codes_referenced(xlsx_paths["haemonc-v1.1.xlsx"], "Haematological Oncology", "Legacy")
cns_legacy_referenced = legacy_m_codes_referenced(xlsx_paths["cancer-cns-v2.xlsx"], "Central Nervous System", "Legacy")
legacy_referenced = haemonc_legacy_referenced | cns_legacy_referenced

not_in_v16 = set(current_m) - set(m_master)
explained = not_in_v16 & legacy_referenced
unexplained = sorted(not_in_v16 - legacy_referenced, key=natural_sort_key)

# category per legacy code - haematological-oncology if Haem Onc v1.1's own Legacy
# column names it, cancer otherwise (see the markdown above for the "otherwise" caveat)
legacy_category = {code: ("haematological-oncology" if code in haemonc_legacy_referenced else "cancer")
                    for code in not_in_v16}

print(f"{len(not_in_v16)} M* codes in the current CodeSystem aren't in v16")
print(f"  {len(explained)} of those are referenced as a legacy code by haem-onc v1.1 or CNS v2 - preserved as-is")
print(f"  {len(unexplained)} are NOT referenced anywhere checked - needs manual follow-up:")
print(" ", unexplained[:30])
print(f"category split: {sum(1 for v in legacy_category.values() if v == 'haematological-oncology')} haematological-oncology, "
      f"{sum(1 for v in legacy_category.values() if v == 'cancer')} cancer")

1180 M* codes in the current CodeSystem aren't in v16
  1166 of those are referenced as a legacy code by haem-onc v1.1 or CNS v2 - preserved as-is
  14 are NOT referenced anywhere checked - needs manual follow-up:
  ['M136.2', 'M136.3', 'M136.5', 'M145.2', 'M153.1', 'M153.4', 'M153.6', 'M156.5', 'M156.7', 'M168.1', 'M168.7', 'M189.4', 'M192.11', 'M193.11']
category split: 546 haematological-oncology, 634 cancer


### Regenerating `EnglandTestCode.fsh`

Three blocks: `R*` and the still-current `M*` rebuilt fresh from `v9`/`v16` (this is
what "periodically rerun" means - each run is a full, deterministic re-render, not a
hand-patched diff), the legacy `M*` block carried over unchanged from the current
file (re-emitted from parsed `(code, display)` pairs, which also fixes the stray-comma
syntax bug found above as a side effect - no special-case handling needed).

Every code carries a `category` `^property` (`#code`-typed: `rare-and-inherited-disease`
/ `cancer` / `haematological-oncology`); R* codes also carry `specialist-test-group` and
`test-method` (both `#string`-typed), and current M* codes carry `test-method` where the
source spreadsheet has a usable value (see the two sections above for exactly what's
populated where and why). These three properties must be declared once at the
`CodeSystem` level (`^property[+].code = ...`) before any concept uses them - the same
pattern the existing `parent` property declaration in `DigitalGenomicTestServices.fsh`
(Step 2, below) already follows.

In [9]:
import datetime

today = datetime.date.today().isoformat()
legacy_m = {k: v for k, v in current_m.items() if k in explained or k in set(unexplained)}
no_test_method_m = sorted((k for k in m_master if k not in m_test_method), key=natural_sort_key)

lines = [
    "CodeSystem: NHSEngland-GenomicTestCode",
    "Id: GenomicTestCode",
    'Title: "NHS England Genomic Test Code"',
    'Description: """',
    "- Source: [National genomic test directories](https://www.england.nhs.uk/publication/national-genomic-test-directories/) -",
    f"  Rare & Inherited Disease v9 and Cancer (non-CNS) v16 checked/refreshed on {today}.",
    "- The M* codes for Haematological Oncology and Cancer CNS below predate those two",
    "  directories' move to the digital TP/GT scheme and no longer appear in any live",
    "  M*-coded master spreadsheet - preserved for backwards compatibility, not",
    "  refreshable from a current source. See `ConceptMap-GenomicTestCodeToDigitalGenomicTestServices`",
    "  for their DGTS GT-code replacements.",
    "- `specialist-test-group` is populated for R* codes only, from the Rare &",
    "  Inherited Disease spreadsheet's own \"Specialist test group\" column - the",
    "  Cancer directories have no equivalent column, so M* codes don't carry it.",
    "- `test-method` is populated for R* codes from that same spreadsheet's \"Test",
    "  Method\" column, and for current (non-legacy) M* codes from the Cancer",
    "  directory's \"Technology\" column (the same concept, differently named) -",
    "  not populated for legacy M* codes, or the small number of current M* codes",
    f"  ({', '.join('`' + k + '`' for k in no_test_method_m)}) whose source cell had no clean single value.",
    "",
    "HL7 v2 CodeSystem : England-GenomicTestDirectory",
    '"""',
    "",
    '* ^name = "GenomicTestCode"',
    "* ^content = #fragment",
    "* ^caseSensitive = true",
    "* ^experimental = false",
    "* ^status = #active",
    f'* ^version = "{today.replace("-", ".")}"',
    f'* ^date = "{today}"',
    '* ^url = "https://fhir.nhs.uk/CodeSystem/England-GenomicTestDirectory"',
    "",
    '* ^property[+].code = #category',
    '* ^property[=].uri = "https://fhir.nhs.uk/CodeSystem/England-GenomicTestDirectory#category"',
    '* ^property[=].description = "Which part of the National Genomic Test Directory this code belongs to: rare-and-inherited-disease, cancer, or haematological-oncology"',
    "* ^property[=].type = #code",
    "",
    '* ^property[+].code = #specialist-test-group',
    '* ^property[=].uri = "https://fhir.nhs.uk/CodeSystem/England-GenomicTestDirectory#specialist-test-group"',
    '* ^property[=].description = "The Specialist test group column from the Rare & Inherited Disease National Genomic Test Directory master spreadsheet (R* codes only - the Cancer directories have no equivalent column)"',
    "* ^property[=].type = #string",
    "",
    '* ^property[+].code = #test-method',
    '* ^property[=].uri = "https://fhir.nhs.uk/CodeSystem/England-GenomicTestDirectory#test-method"',
    '* ^property[=].description = "The test method/technology used - Test Method column (R* codes) or Technology column (current M* codes); not populated for legacy M* codes"',
    "* ^property[=].type = #string",
    "",
    "// R* - Rare & Inherited Disease (v9)",
    "",
]
for k in sorted(r_master, key=natural_sort_key):
    lines.append(f'* #{k} "{fsh_escape(r_master[k])}"')
    lines.append("  * ^property[+].code = #category")
    lines.append("  * ^property[=].valueCode = #rare-and-inherited-disease")
    if k in r_specialist_group:
        lines.append("  * ^property[+].code = #specialist-test-group")
        lines.append(f'  * ^property[=].valueString = "{fsh_escape(r_specialist_group[k])}"')
    if k in r_test_method:
        lines.append("  * ^property[+].code = #test-method")
        lines.append(f'  * ^property[=].valueString = "{fsh_escape(r_test_method[k])}"')

lines += ["", "// M* - Cancer, non-CNS (v16) - still current", ""]
for k in sorted(m_master, key=natural_sort_key):
    lines.append(f'* #{k} "{fsh_escape(m_master[k])}"')
    lines.append("  * ^property[+].code = #category")
    lines.append("  * ^property[=].valueCode = #cancer")
    if k in m_test_method:
        lines.append("  * ^property[+].code = #test-method")
        lines.append(f'  * ^property[=].valueString = "{fsh_escape(m_test_method[k])}"')

lines += ["", "// M* - Haematological Oncology / Cancer CNS - superseded by DGTS GT codes, preserved as-is", ""]
for k in sorted(legacy_m, key=natural_sort_key):
    lines.append(f'* #{k} "{fsh_escape(legacy_m[k])}"')
    lines.append("  * ^property[+].code = #category")
    lines.append(f"  * ^property[=].valueCode = #{legacy_category[k]}")

england_test_code_fsh = "\n".join(lines) + "\n"

out_path = os.path.join(FSH_CODESYSTEM_DIR, "EnglandTestCode.fsh")
with open(out_path, "w") as f:
    f.write(england_test_code_fsh)

print(f"wrote {out_path}")
print(f"{len(r_master) + len(m_master) + len(legacy_m)} total codes ({len(r_master)} R*, {len(m_master)} current M*, {len(legacy_m)} legacy M*)")
print(f"properties: category on all of them; specialist-test-group on {len(r_specialist_group)}; "
      f"test-method on {len(r_test_method) + len(m_test_method)} ({len(no_test_method_m)} current M* code(s) with no usable value: {no_test_method_m})")

wrote /Users/kevinmayfield/github/MFT/FHIRGenomics/input/fsh/Termininology/CodeSystem/EnglandTestCode.fsh
2130 total codes (457 R*, 493 current M*, 1180 legacy M*)
properties: category on all of them; specialist-test-group on 457; test-method on 948 (2 current M* code(s) with no usable value: ['M4.14', 'M119.5'])


### `GenomicClinicalIndication`: completing the 1st-level codes

`ClinicalIndication.fsh` (`GenomicClinicalIndication`) is a hand-picked example
fragment today - 17 codes, one (`M4`) with no display text at all. Both master
spreadsheets carry the 1st-level clinical-indication columns this CodeSystem is
supposed to hold in full: `Clinical indication ID`/`Clinical Indication` in `v9`,
`CI code`/`Clinical indication name` in `v16` - same forward-fill reasoning as the
test-level `M*` codes above. Checked first: does a given CI code's name actually stay
consistent across every test row under it? (Yes, for both - 378 R* and 114 M* CI
codes, zero inconsistencies). Two of the existing fragment's codes (`R362`, `R94`,
both `"Not present in 8.0"`) are in neither master file - retired CI numbers, not a
gap in this refresh - so a first pass here dropped them entirely; fixed the same way
`EnglandTestCode.fsh` already handles the equivalent case, by preserving anything the
old fragment carries that isn't accounted for by a live source.

In [10]:
wb = openpyxl.load_workbook(xlsx_paths["rare-v9.xlsx"], data_only=True)
ws = wb["R&ID indications"]
r_ci_master = {}
for row in range(2, ws.max_row + 1):
    ci_id = ws.cell(row=row, column=1).value
    ci_name = ws.cell(row=row, column=3).value
    if ci_id:
        r_ci_master[ci_id.strip()] = ci_name.strip()

wb = openpyxl.load_workbook(xlsx_paths["cancer-noncns-v16.xlsx"], data_only=True)
m_ci_master = {}
for sheet in ["Solid tumours", "Sarcoma", "Paediatric"]:
    ws = wb[sheet]
    for row in range(3, ws.max_row + 1):
        ci_id = ws.cell(row=row, column=3).value
        ci_name = ws.cell(row=row, column=4).value
        if ci_id:
            m_ci_master[ci_id.strip()] = ci_name.strip()

print(f"{len(r_ci_master)} R* CI codes (v9), {len(m_ci_master)} M* CI codes (v16, still current)")

378 R* CI codes (v9), 114 M* CI codes (v16, still current)


In [11]:
def shared_leading_text(test_codes_and_displays, ci_number):
    names = {d.split(",", 1)[0].strip() for k, d in test_codes_and_displays.items() if k.split(".")[0] == ci_number}
    return sorted(names, key=len)[0] if names else ci_number


legacy_ci_numbers = {k.split(".")[0] for k in legacy_m}
legacy_ci_master = {ci: shared_leading_text(legacy_m, ci) for ci in legacy_ci_numbers}

print(f"{len(legacy_ci_master)} legacy M* CI numbers (Haematological Oncology / Cancer CNS) - display text")
print("derived from EnglandTestCode.fsh's own legacy M*.* entries, same reasoning as Step 3 below.")

103 legacy M* CI numbers (Haematological Oncology / Cancer CNS) - display text
derived from EnglandTestCode.fsh's own legacy M*.* entries, same reasoning as Step 3 below.


In [12]:
existing_ci_text = fetch_current_fsh("ClinicalIndication.fsh")
existing_ci_codes = parse_fsh_codes(existing_ci_text)

# Codes the existing fragment carries that aren't in v9/v16 at all any more (e.g.
# "Not present in 8.0" tombstones for a retired CI number) - preserved, not dropped,
# same reasoning as the legacy M* test-code block in EnglandTestCode.fsh above.
retired_ci = {k: v for k, v in existing_ci_codes.items() if k not in r_ci_master and k not in m_ci_master and k not in legacy_ci_master}

all_ci = {**r_ci_master, **m_ci_master, **legacy_ci_master, **retired_ci}
ci_added = sorted(set(all_ci) - set(existing_ci_codes), key=natural_sort_key)
ci_changed = sorted((k for k in set(all_ci) & set(existing_ci_codes) if all_ci[k] != existing_ci_codes[k]), key=natural_sort_key)
print(f"{len(existing_ci_codes)} codes today -> {len(all_ci)} once completed ({len(ci_added)} new)")
print(f"{len(retired_ci)} code(s) in neither master file, preserved as retired: {retired_ci}")
print(f"{len(ci_changed)} of the {len(set(all_ci) & set(existing_ci_codes))} already-present codes have different display text:")
for k in ci_changed:
    print(f"  {k}: was {existing_ci_codes[k]!r}, now {all_ci[k]!r}")

597 codes today -> 597 once completed (0 new)
2 code(s) in neither master file, preserved as retired: {'R94': 'Not present in 8.0', 'R362': 'Not present in 8.0'}
0 of the 597 already-present codes have different display text:


In [13]:
lines = [
    "CodeSystem: GenomicClinicalIndication",
    "Id: GenomicClinicalIndication",
    'Title: "NHS England Genomic Clinical Indication Code"',
    'Description: """',
    "1st level Genomic Test Directory Codes - completed from the same master",
    "spreadsheets as `GenomicTestCode` (`EnglandTestCode.fsh`); see that CodeSystem's",
    "own Description for the legacy-versus-current distinction, which applies the",
    "same way here.",
    '"""',
    "",
    '* ^name = "GenomicClinicalIndication"',
    "* ^content = #fragment",
    "* ^caseSensitive = true",
    "* ^experimental = false",
    "* ^status = #active",
    f'* ^version = "{today.replace("-", ".")}"',
    f'* ^date = "{today}"',
    '* ^url = "https://fhir.nwgenomics.nhs.uk/CodeSystem/GenomicClinicalIndication"',
    "",
    "// R* - Rare & Inherited Disease (v9)",
    "",
]
for k in sorted(r_ci_master, key=natural_sort_key):
    lines.append(f'* #{k} "{fsh_escape(r_ci_master[k])}"')

lines += ["", "// M* - Cancer, non-CNS (v16) - still current", ""]
for k in sorted(m_ci_master, key=natural_sort_key):
    lines.append(f'* #{k} "{fsh_escape(m_ci_master[k])}"')

lines += ["", "// M* - Haematological Oncology / Cancer CNS - superseded by DGTS TP codes, preserved as-is", ""]
for k in sorted(legacy_ci_master, key=natural_sort_key):
    lines.append(f'* #{k} "{fsh_escape(legacy_ci_master[k])}"')

lines += ["", "// Retired - not in v9/v16, preserved from the existing fragment rather than dropped", ""]
for k in sorted(retired_ci, key=natural_sort_key):
    lines.append(f'* #{k} "{fsh_escape(retired_ci[k])}"')

clinical_indication_fsh = "\n".join(lines) + "\n"
out_path = os.path.join(FSH_CODESYSTEM_DIR, "ClinicalIndication.fsh")
with open(out_path, "w") as f:
    f.write(clinical_indication_fsh)

print(f"wrote {out_path}")
print(f"{len(all_ci)} total CI codes ({len(r_ci_master)} R*, {len(m_ci_master)} current M*, {len(legacy_ci_master)} legacy M*, {len(retired_ci)} retired)")

wrote /Users/kevinmayfield/github/MFT/FHIRGenomics/input/fsh/Termininology/CodeSystem/ClinicalIndication.fsh
597 total CI codes (378 R*, 114 current M*, 103 legacy M*, 2 retired)


### `GenomicClinicalIndication` → `GenomicTestCode`: a `narrower` `ConceptMap`

Both CodeSystems are complete now, so the relationship between them is just pattern
matching: every `GenomicTestCode` entry `<CI>.<n>` belongs to `GenomicClinicalIndication`
entry `<CI>`. The target (a specific test) is always narrower in scope than the source
(the clinical indication it sits under) - `equivalence = #narrower` throughout, not the
`equivalent`/`relatedto` split Step 3 uses below, since this relationship is
structural, not empirical.

In [14]:
all_test_codes = {**r_master, **m_master, **legacy_m}

ci_to_tests = defaultdict(set)
for test_code in all_test_codes:
    ci_number = test_code.split(".")[0]
    if ci_number in all_ci:
        ci_to_tests[ci_number].add(test_code)

orphaned_tests = set(all_test_codes) - {t for tests in ci_to_tests.values() for t in tests}
print(f"{len(ci_to_tests)} of {len(all_ci)} CI codes have at least one matching test code")
print(f"{len(orphaned_tests)} test code(s) whose CI-number prefix isn't in GenomicClinicalIndication at all:")
print(" ", sorted(orphaned_tests, key=natural_sort_key)[:15])

595 of 597 CI codes have at least one matching test code
0 test code(s) whose CI-number prefix isn't in GenomicClinicalIndication at all:
  []


In [15]:
narrower_conceptmap_fsh = build_concept_map_fsh(
    instance_name="GenomicClinicalIndicationToGenomicTestCode",
    title="Genomic Clinical Indication to Genomic Test Code (narrower)",
    description=(
        "Every `GenomicTestCode` (`England-GenomicTestDirectory`) entry belongs to\n"
        "exactly one `GenomicClinicalIndication` entry, by construction (`<CI>.<n>`\n"
        "under `<CI>`) - not an empirical mapping like the legacy/digital ConceptMaps\n"
        "below, so every element uses `equivalence = #narrower` (the target, a specific\n"
        "test, is always narrower in scope than the source, the clinical indication it\n"
        "sits under), regardless of how many tests a given indication has.\n\n"
        "Built by [nw-gmsa/Testing notebook 14](https://github.com/nw-gmsa/Testing/blob/main/notebooks/14-national-genomic-test-directory-codesystems.ipynb),"
        f" checked against v9/v16 on {today}."
    ),
    source_url="https://fhir.nwgenomics.nhs.uk/CodeSystem/GenomicClinicalIndication",
    source_version=today,
    target_url="https://fhir.nhs.uk/CodeSystem/England-GenomicTestDirectory",
    mapping=ci_to_tests,
    target_display=all_test_codes,
    source_display=all_ci,
    fixed_equivalence="narrower",
)

out_path = os.path.join(FSH_CONCEPTMAP_DIR, "GenomicClinicalIndicationToGenomicTestCode.fsh")
with open(out_path, "w") as f:
    f.write(narrower_conceptmap_fsh)
print(f"wrote {out_path} - {len(ci_to_tests)} elements, {sum(len(v) for v in ci_to_tests.values())} total targets")

wrote /Users/kevinmayfield/github/MFT/FHIRGenomics/input/fsh/Termininology/ConceptMap/GenomicClinicalIndicationToGenomicTestCode.fsh - 595 elements, 2130 total targets


## Step 2: `DigitalGenomicTestServices`

Two directories have already gone fully digital: Haematological Oncology (v1.1) and
Cancer CNS (v2). Both share the same idea, slightly different header wording - `Test
package ID`/`Test package name` group into a **TP** code, `Genomic test (GT) code`/
`Test name` identify the specific investigation - matched by header text rather than
fixed column positions, since the two files don't agree on exact column order/casing.

In [16]:
def find_column(header, *needles):
    for i, h in enumerate(header):
        if not h:
            continue
        normalised = " ".join(h.lower().split())
        if all(n in normalised for n in needles):
            return i + 1
    raise KeyError(needles)


tp_codes = {}
gt_codes = {}
gt_conflicts = []
gt_to_tp = defaultdict(set)
gt_test_method = {}
gt_tm_conflicts = []
tp_source_haemonc = set()  # for the category default below - which file each code came from
gt_source_haemonc = set()

for filename, sheet in [("haemonc-v1.1.xlsx", "Haematological Oncology"), ("cancer-cns-v2.xlsx", "Central Nervous System")]:
    is_haemonc = filename == "haemonc-v1.1.xlsx"
    wb = openpyxl.load_workbook(xlsx_paths[filename], data_only=True)
    ws = wb[sheet]
    header = [ws.cell(row=2, column=c).value for c in range(1, ws.max_column + 1)]
    c_tp, c_tpname = find_column(header, "test package id"), find_column(header, "test package name")
    c_gt, c_gtname = find_column(header, "genomic test", "code"), find_column(header, "test name")
    c_tm = find_column(header, "test method")

    for row in range(3, ws.max_row + 1):
        tp_id = ws.cell(row=row, column=c_tp).value
        tp_name = ws.cell(row=row, column=c_tpname).value
        gt_id = ws.cell(row=row, column=c_gt).value
        gt_name = ws.cell(row=row, column=c_gtname).value
        test_method = ws.cell(row=row, column=c_tm).value
        if tp_id:
            tp_id = tp_id.strip()
            tp_codes[tp_id] = tp_name.strip()
            if is_haemonc:
                tp_source_haemonc.add(tp_id)
        if gt_id:
            gt_id, gt_name = gt_id.strip(), " ".join(gt_name.split())
            if gt_id in gt_codes and gt_codes[gt_id] != gt_name:
                gt_conflicts.append((filename, gt_id, gt_codes[gt_id], gt_name))
            gt_codes[gt_id] = gt_name
            if is_haemonc:
                gt_source_haemonc.add(gt_id)
            if tp_id:
                gt_to_tp[gt_id].add(tp_id)
            if test_method:
                test_method = str(test_method).strip()
                if gt_id in gt_test_method and gt_test_method[gt_id] != test_method:
                    gt_tm_conflicts.append((filename, gt_id, gt_test_method[gt_id], test_method))
                gt_test_method[gt_id] = test_method

gt_to_tp_multi = {k: v for k, v in gt_to_tp.items() if len(v) > 1}
print(f"{len(tp_codes)} TP codes, {len(gt_codes)} GT codes")
print(f"{len(gt_to_tp_multi)} of {len(gt_to_tp)} GT codes belong to more than one TP (not a strict tree - modelled")
print("as a repeatable 'parent' property below, per http://hl7.org/fhir/codesystem.html#hierarchy, not nested concepts)")
print(f"{len(gt_conflicts)} GT code(s) with inconsistent whitespace/casing across rows (resolved to the last-seen form):")
for c in gt_conflicts:
    print(" ", c)
print(f"{len(gt_test_method)} of {len(gt_codes)} GT codes have a Test Method value")
print(f"{len(gt_tm_conflicts)} GT code(s) with an inconsistent Test Method across rows (resolved to the last-seen form):")
for c in gt_tm_conflicts:
    print(" ", c)

21 TP codes, 352 GT codes


47 of 352 GT codes belong to more than one TP (not a strict tree - modelled
as a repeatable 'parent' property below, per http://hl7.org/fhir/codesystem.html#hierarchy, not nested concepts)
3 GT code(s) with inconsistent whitespace/casing across rows (resolved to the last-seen form):
  ('cancer-cns-v2.xlsx', 'GT453', 'RELA rearrangement - FISH', 'RELA rearrangement FISH')
  ('cancer-cns-v2.xlsx', 'GT592', 'BRAF::KIAA1549 Rearrangement', 'BRAF::KIAA1549 rearrangement')
  ('cancer-cns-v2.xlsx', 'GT1280', 'MN1 rearrangement FISH', 'MN1 rearrangement')
352 of 352 GT codes have a Test Method value
1 GT code(s) with an inconsistent Test Method across rows (resolved to the last-seen form):
  ('cancer-cns-v2.xlsx', 'GT1280', 'Targeted assay', 'FISH')


### Regenerating `DigitalGenomicTestServices.fsh`

The existing `TP171`/`GT497` ("Cystic renal disease") pairing isn't sourced from
either of these two files (neither haem-onc nor CNS names it) - some other,
already-migrated indication this notebook doesn't cover. Preserved as-is, with the new
codes from both files appended - and, since both share the one name, given the same
`TP171` parent as everything else below.

Each `GT` concept gets its `TP`(s) recorded as a `parent` `property`
(`http://hl7.org/fhir/concept-properties#parent`) rather than nested `concept`s - FHIR's
own documented pattern for a hierarchy that isn't a strict tree, which this one isn't:
47 `GT` codes above answered to more than one `TP`, so any single `GT` can carry more
than one `parent` property instance.

Two more properties, on both `TP` and `GT` concepts:

- **`category`** - which part of the Test Directory a code belongs to. Mostly
  file-derived (haem-onc v1.1 rows → `haematological-oncology`, CNS v2 rows →
  `cancer`), but not *entirely*: `TP401` "Chimerism Testing" is a haem-onc-file row
  categorised `chimerism` instead, and the preserved `TP171`/`GT497` pair (sourced from
  neither file) is categorised `rare-and-inherited-disease` - both curated by hand
  outside this notebook, not derivable from any column here. So rather than
  re-deriving `category` from the source files alone (which would silently overwrite
  `TP401`'s override back to `haematological-oncology`, and drop `TP171`/`GT497`'s
  entirely, on every rerun), this step **preserves whatever `category` a code already
  carries in the currently-published FSH** (`parse_fsh_categories()` below), and only
  falls back to the file-derived default for a code that doesn't have one yet - i.e.
  a genuinely new `TP`/`GT` this run has just discovered. A newly-added override still
  needs a human to add it (the same "notebook refreshes, a person curates the
  exceptions" split already used for `TP171`/`GT497` themselves) - this step's own
  summary output flags any `TP`/`GT` that ends up with no `category` at all, so that
  need is visible rather than silent.
- **`test-method`** - `GT` concepts only (a `TP` package can legitimately bundle more
  than one technology, so it doesn't get one fixed value) - taken directly from each
  source row's own "Test Method" column, the same concept `EnglandTestCode.fsh`'s R*/M*
  `test-method` property already draws from its own directories. Unlike `category`,
  this one **is** cleanly re-derivable from source every run, so it isn't preserved the
  same way - a genuine value change in a future spreadsheet version should flow
  through, not get stuck on whatever an earlier run recorded.

No `specialist-test-group` here: neither the Haematological Oncology nor the Cancer CNS
master spreadsheet carries an equivalent column at all (unlike Rare & Inherited
Disease's own "Specialist test group" column, which `EnglandTestCode.fsh`'s R* codes
use) - there's nothing to source it from, so it isn't declared on this `CodeSystem`
rather than being declared and left permanently empty.

In [17]:
existing_dgts_text = fetch_current_fsh("DigitalGenomicTestServices.fsh")
existing_dgts_codes = parse_fsh_codes(existing_dgts_text)
existing_dgts_categories = parse_fsh_categories(existing_dgts_text)
preserved_tp = {k: v for k, v in existing_dgts_codes.items() if k.startswith("TP") and k not in tp_codes}
preserved_gt = {k: v for k, v in existing_dgts_codes.items() if k.startswith("GT") and k not in gt_codes}
print("Preserved (not sourced from haem-onc/CNS):", preserved_tp, preserved_gt)

for gt_id, tp_id in [("GT497", "TP171")]:  # preserved pair - both named "Cystic renal disease", not in gt_to_tp from source data
    if gt_id in preserved_gt and tp_id in preserved_tp:
        gt_to_tp[gt_id].add(tp_id)


def category_for(code, is_haemonc_source):
    """Preserve whatever category the currently-published FSH already records
    for this code; only fall back to the file-derived default (haematological-
    oncology / cancer) for a code that's never had one recorded - i.e. brand
    new this run. Returns None if there's simply nothing to go on (a code not
    in the current file, not sourced from either directory - shouldn't happen
    for anything in tp_codes/gt_codes, but preserved_tp/preserved_gt entries
    with no prior category would land here too, and get no category property
    at all rather than a guessed one."""
    if code in existing_dgts_categories:
        return existing_dgts_categories[code]
    if is_haemonc_source is None:
        return None
    return "haematological-oncology" if is_haemonc_source else "cancer"


lines = [
    "CodeSystem: NHSEngland-DigitalGenomicTestServices",
    "Id: DigitalGenomicTestServices",
    'Title: "NHS England Digital Genomic Test Services"',
    'Description: """',
    "The **digital** National Genomic Test Directory codes, rolled out as the",
    "Test Directory itself went digital - replacing the legacy R-code/M-code",
    "style codes in [NHS England Genomic Test",
    "Code](CodeSystem-GenomicTestCode.html) (`$GTD`,",
    "`England-GenomicTestDirectory`) for new/migrated indications. Two code",
    "types share this one system, distinguished by prefix:",
    "",
    "- **TP (Test Package)** codes replace the old clinical-indication concept -",
    "  a package of related genomic tests grouped by clinical condition.",
    "- **GT (Genomic/Genetic Test)** codes identify a single specific laboratory",
    "  investigation.",
    "",
    "Refreshed from the Haematological Oncology (v1.1) and Cancer CNS (v2) national",
    f"genomic test directories on {today} - see `ConceptMap-GenomicTestCodeToDigitalGenomicTestServices`",
    "and `ConceptMap-GenomicClinicalIndicationToDigitalGenomicTestServices` for how",
    "these relate to the legacy codes they replace.",
    "",
    "`category` is mostly file-derived (haem-onc v1.1 -> haematological-oncology, CNS",
    "v2 -> cancer) but not entirely - a handful of codes are curated by hand outside",
    "this notebook (e.g. `TP401` \"Chimerism Testing\" -> chimerism) and preserved",
    "across reruns rather than re-derived; see notebook 14's own note on this for the",
    "full reasoning. `test-method` (GT codes only) comes straight from each source",
    "row's own \"Test Method\" column, refreshed fresh every run. No",
    "`specialist-test-group` here - neither source directory has an equivalent column.",
    '"""',
    "",
    '* ^name = "DigitalGenomicTestServices"',
    "* ^content = #fragment",
    "* ^caseSensitive = true",
    "* ^experimental = false",
    "* ^status = #active",
    f'* ^version = "{today.replace("-", ".")}"',
    f'* ^date = "{today}"',
    '* ^url = "https://fhir.nhs.uk/CodeSystem/England-DigitalGenomicTestServices"',
    "",
    '* ^property[+].code = #parent',
    '* ^property[=].uri = "http://hl7.org/fhir/concept-properties#parent"',
    '* ^property[=].description = "The Test Package (TP) code(s) this Genomic Test (GT) code belongs to"',
    "* ^property[=].type = #code",
    "",
    '* ^property[+].code = #category',
    '* ^property[=].uri = "https://fhir.nhs.uk/CodeSystem/England-DigitalGenomicTestServices#category"',
    '* ^property[=].description = "Which part of the National Genomic Test Directory this code belongs to: rare-and-inherited-disease, cancer, haematological-oncology, or chimerism"',
    "* ^property[=].type = #code",
    "",
    '* ^property[+].code = #test-method',
    '* ^property[=].uri = "https://fhir.nhs.uk/CodeSystem/England-DigitalGenomicTestServices#test-method"',
    '* ^property[=].description = "The test method/technology used, from the source directory\'s own Test Method column (Genomic/Genetic Test (GT) codes only - a Test Package (TP) can bundle more than one technology)"',
    "* ^property[=].type = #string",
    "",
    "// Test Package (TP) - clinical-indication-level grouping",
    "",
]

tp_no_category = []
for k in sorted({**preserved_tp, **tp_codes}, key=natural_sort_key):
    display = tp_codes.get(k, preserved_tp.get(k))
    lines.append(f'* #{k} "{fsh_escape(display)}"')
    is_haemonc_source = (k in tp_source_haemonc) if k in tp_codes else None
    category = category_for(k, is_haemonc_source)
    if category:
        lines.append("  * ^property[+].code = #category")
        lines.append(f"  * ^property[=].valueCode = #{category}")
    else:
        tp_no_category.append(k)

lines += ["", "// Genomic/Genetic Test (GT) - specific laboratory investigation", ""]
gt_no_category = []
for k in sorted({**preserved_gt, **gt_codes}, key=natural_sort_key):
    display = gt_codes.get(k, preserved_gt.get(k))
    lines.append(f'* #{k} "{fsh_escape(display)}"')
    is_haemonc_source = (k in gt_source_haemonc) if k in gt_codes else None
    category = category_for(k, is_haemonc_source)
    if category:
        lines.append("  * ^property[+].code = #category")
        lines.append(f"  * ^property[=].valueCode = #{category}")
    else:
        gt_no_category.append(k)
    if k in gt_test_method:
        lines.append("  * ^property[+].code = #test-method")
        lines.append(f'  * ^property[=].valueString = "{fsh_escape(gt_test_method[k])}"')
    for parent_tp in sorted(gt_to_tp.get(k, ())):
        lines.append("  * ^property[+].code = #parent")
        lines.append(f"  * ^property[=].valueCode = #{parent_tp}")

dgts_fsh = "\n".join(lines) + "\n"
out_path = os.path.join(FSH_CODESYSTEM_DIR, "DigitalGenomicTestServices.fsh")
with open(out_path, "w") as f:
    f.write(dgts_fsh)

all_gt_codes_written = {**preserved_gt, **gt_codes}
gt_without_parent = sorted((k for k in all_gt_codes_written if k not in gt_to_tp), key=natural_sort_key)
print(f"wrote {out_path}")
print(f"{len(preserved_tp) + len(tp_codes)} TP codes, {len(all_gt_codes_written)} GT codes")
print(f"{len(gt_without_parent)} GT code(s) with no `parent` property at all: {gt_without_parent}")
print(f"{len(gt_test_method)} GT code(s) with a test-method value")
if tp_no_category or gt_no_category:
    print(f"NEEDS MANUAL CURATION - no category at all for {len(tp_no_category)} TP, {len(gt_no_category)} GT: "
          f"{tp_no_category + gt_no_category}")
else:
    print("Every TP/GT code has a category (preserved from the current file or file-derived).")

Preserved (not sourced from haem-onc/CNS): {'TP171': 'Cystic renal disease'} {'GT497': 'Cystic renal disease - WGS'}
wrote /Users/kevinmayfield/github/MFT/FHIRGenomics/input/fsh/Termininology/CodeSystem/DigitalGenomicTestServices.fsh
22 TP codes, 353 GT codes
0 GT code(s) with no `parent` property at all: []
352 GT code(s) with a test-method value
Every TP/GT code has a category (preserved from the current file or file-derived).


## Step 3: `ConceptMap`s from legacy to digital

Both haem-onc and CNS list a `Legacy 'M' codes' column per `GT` row, and a `Test
package ID` per row too - which means both halves of this mapping are given directly
by NHS England's own spreadsheets, not inferred:

- `M*` test code → `GT*` code, straight off the `Legacy 'M' codes` column.
- Old clinical-indication number (`M100`, `M90`, ...) → `TP*` code, derived by taking
  each legacy `M*` code's leading number and looking at which `TP` its row belongs to.

Neither direction turns out to be simple 1:1, though - worth showing honestly rather
than forcing it.

In [18]:
m_to_gt = defaultdict(set)
ci_to_tp = defaultdict(set)

for filename, sheet in [("haemonc-v1.1.xlsx", "Haematological Oncology"), ("cancer-cns-v2.xlsx", "Central Nervous System")]:
    wb = openpyxl.load_workbook(xlsx_paths[filename], data_only=True)
    ws = wb[sheet]
    header = [ws.cell(row=2, column=c).value for c in range(1, ws.max_column + 1)]
    c_tp = find_column(header, "test package id")
    c_gt = find_column(header, "genomic test", "code")
    c_legacy = find_column(header, "legacy")

    for row in range(3, ws.max_row + 1):
        tp_id = ws.cell(row=row, column=c_tp).value
        gt_id = ws.cell(row=row, column=c_gt).value
        legacy = ws.cell(row=row, column=c_legacy).value
        if not gt_id or not legacy:
            continue
        gt_id = gt_id.strip()
        for legacy_code in str(legacy).split(","):
            legacy_code = legacy_code.strip()
            if not legacy_code:
                continue
            m_to_gt[legacy_code].add(gt_id)
            if tp_id:
                ci_to_tp[legacy_code.split(".")[0]].add(tp_id.strip())

m_to_gt_multi = {k: v for k, v in m_to_gt.items() if len(v) > 1}
ci_to_tp_multi = {k: v for k, v in ci_to_tp.items() if len(v) > 1}
print(f"{len(m_to_gt)} legacy M* test codes map to a GT code - {len(m_to_gt_multi)} of them to more than one")
print(f"{len(ci_to_tp)} old CI numbers map to a TP code - {len(ci_to_tp_multi)} of them to more than one")

not_in_current_fsh = sorted((k for k in m_to_gt if k not in current_m), key=natural_sort_key)
print(f"{len(not_in_current_fsh)} legacy code(s) named in the haem-onc/CNS spreadsheets that were never in")
print(f"  EnglandTestCode.fsh at all - a gap in the old FSH or a typo in NHS England's own")
print(f"  spreadsheet, not something this notebook can resolve: {not_in_current_fsh[:15]}")
print()
print("Example one-to-many (M-code shared by multiple GT-coded tests):")
k = next(iter(m_to_gt_multi))
print(f"  {k} -> {m_to_gt_multi[k]}")
print("Example one-to-many (old CI grouped a shared assay used across several new TPs):")
k = next(iter(ci_to_tp_multi))
print(f"  {k} -> {ci_to_tp_multi[k]}")

1304 legacy M* test codes map to a GT code - 70 of them to more than one
175 old CI numbers map to a TP code - 136 of them to more than one
11 legacy code(s) named in the haem-onc/CNS spreadsheets that were never in
  EnglandTestCode.fsh at all - a gap in the old FSH or a typo in NHS England's own
  spreadsheet, not something this notebook can resolve: ['M85.39', 'M246.1', 'M246.2', 'M246.3', 'M246.4', 'M247.1', 'M247.2', 'M247.3', 'M247.4', 'R428.1', 'R428.2']

Example one-to-many (M-code shared by multiple GT-coded tests):
  M89.7 -> {'GT329', 'GT185'}
Example one-to-many (old CI grouped a shared assay used across several new TPs):
  M90 -> {'TP241', 'TP450', 'TP374', 'TP34'}


### Where the old CI display text comes from

There's no live master file for the old haem-onc/CNS clinical-indication grouping any
more (same reasoning as Step 1's legacy `M*` block) - the only surviving record of
what e.g. `M100` was actually called is the current `EnglandTestCode.fsh` itself.
Derived from the shared leading text (before the first comma) of that CI number's own
`M*.*` entries.

In [19]:
def ci_display_name(ci_number):
    names = {current_m[k].split(",", 1)[0].strip() for k in current_m if k.split(".")[0] == ci_number}
    return sorted(names, key=len)[0] if names else ci_number  # shortest shared prefix if they don't all agree exactly


ci_display = {ci: ci_display_name(ci) for ci in ci_to_tp}
inconsistent_ci_names = [ci for ci in ci_to_tp if len({current_m[k].split(",", 1)[0].strip() for k in current_m if k.split(".")[0] == ci}) > 1]
print(f"{len(inconsistent_ci_names)} CI number(s) whose M*.* entries don't all share the exact same leading text:")
for ci in inconsistent_ci_names[:5]:
    print(" ", ci, "->", {current_m[k].split(",", 1)[0].strip() for k in current_m if k.split(".")[0] == ci})

2 CI number(s) whose M*.* entries don't all share the exact same leading text:
  M95 -> {'B cell Non-Hodgkin Lymphoma', 'B Cell Non-Hodgkin Lymphoma'}
  M26 -> {'Ependymoma', 'Ependymoma Supratentorial'}


In [20]:
all_gt_display = {**gt_codes, **preserved_gt}
m_conceptmap_fsh = build_concept_map_fsh(
    instance_name="GenomicTestCodeToDigitalGenomicTestServices",
    title="Genomic Test Code (M*) to Digital Genomic Test Services (GT)",
    description=(
        "Legacy `M*` test codes (`GenomicTestCode`/`England-GenomicTestDirectory`) for\n"
        "Haematological Oncology and Cancer CNS, mapped to the `GT*` codes that replace\n"
        "them - taken directly from the `Legacy 'M' codes` column in NHS England's\n"
        f"Haematological Oncology (v1.1) and Cancer CNS (v2) national genomic test\n"
        "directories, not inferred. Not 1:1 throughout: several old `M*` tests were\n"
        "consolidated into one `GT` code (`equivalence = #equivalent`), and a smaller\n"
        "number split or shared across more than one `GT` code, recorded as multiple\n"
        "`target`s with `equivalence = #relatedto` rather than an arbitrary pick.\n\n"
        "Built by [nw-gmsa/Testing notebook 14](https://github.com/nw-gmsa/Testing/blob/main/notebooks/14-national-genomic-test-directory-codesystems.ipynb),"
        f" checked against v1.1/v2 on {today}."
    ),
    source_url="https://fhir.nhs.uk/CodeSystem/England-GenomicTestDirectory",
    source_version=today,
    target_url="https://fhir.nhs.uk/CodeSystem/England-DigitalGenomicTestServices",
    mapping=m_to_gt,
    target_display=all_gt_display,
    source_display=current_m,
)

out_path = os.path.join(FSH_CONCEPTMAP_DIR, "GenomicTestCodeToDigitalGenomicTestServices.fsh")
with open(out_path, "w") as f:
    f.write(m_conceptmap_fsh)
print(f"wrote {out_path} - {len(m_to_gt)} elements")

wrote /Users/kevinmayfield/github/MFT/FHIRGenomics/input/fsh/Termininology/ConceptMap/GenomicTestCodeToDigitalGenomicTestServices.fsh - 1304 elements


In [21]:
all_tp_display = {**tp_codes, **preserved_tp}
ci_conceptmap_fsh = build_concept_map_fsh(
    instance_name="GenomicClinicalIndicationToDigitalGenomicTestServices",
    title="Genomic Clinical Indication (M*) to Digital Genomic Test Services (TP)",
    description=(
        "Legacy 1st-level clinical-indication numbers (`GenomicClinicalIndication`) for\n"
        "Haematological Oncology and Cancer CNS, mapped to the `TP*` codes that replace\n"
        "them - derived from the same `Legacy 'M' codes` column used for the `GT*` map\n"
        "above (each legacy code's leading number, cross-referenced against which `TP`\n"
        "its row belongs to), since NHS England's spreadsheets don't name the old CI\n"
        "number directly. Genuinely many-to-many for a large share of these: the old\n"
        "haem-onc CI grouping was organised around a shared assay/test method, not\n"
        "disease, so one old CI number routinely feeds several disease-specific `TP`s -\n"
        "recorded as multiple `target`s with `equivalence = #relatedto`, not forced to\n"
        "one.\n\n"
        "Display text for the old CI numbers has no live source any more (see notebook\n"
        "14's own note on this) - taken from the current `EnglandTestCode.fsh`'s own\n"
        "`M*.*` entries for that number.\n\n"
        "Built by [nw-gmsa/Testing notebook 14](https://github.com/nw-gmsa/Testing/blob/main/notebooks/14-national-genomic-test-directory-codesystems.ipynb),"
        f" checked against v1.1/v2 on {today}."
    ),
    source_url="https://fhir.nwgenomics.nhs.uk/CodeSystem/GenomicClinicalIndication",
    source_version=today,
    target_url="https://fhir.nhs.uk/CodeSystem/England-DigitalGenomicTestServices",
    mapping=ci_to_tp,
    target_display=all_tp_display,
    source_display=ci_display,
)

out_path = os.path.join(FSH_CONCEPTMAP_DIR, "GenomicClinicalIndicationToDigitalGenomicTestServices.fsh")
with open(out_path, "w") as f:
    f.write(ci_conceptmap_fsh)
print(f"wrote {out_path} - {len(ci_to_tp)} elements")

wrote /Users/kevinmayfield/github/MFT/FHIRGenomics/input/fsh/Termininology/ConceptMap/GenomicClinicalIndicationToDigitalGenomicTestServices.fsh - 175 elements


## Summary

- `GenomicTestCode` refreshed from live NHS England spreadsheets: `R*` (v9) essentially
  matched already (1 new test, 116 display-text refinements - mostly test-method
  renames and a `UTF-8` mis-decode bug fixed); `M*` split cleanly into "still live in
  v16" (refreshed, a handful of new/changed) versus "predates the haem-onc/CNS move to
  digital" (preserved, 98%+ independently confirmed via those directories' own
  `Legacy 'M' codes` column - not just assumed). Purely additive - zero codes removed.
- Every `GenomicTestCode` concept now also carries a `category` `^property`
  (`rare-and-inherited-disease` / `cancer` / `haematological-oncology` - fixed for R*
  and current M*, per-code for legacy M* depending on which predecessor directory's own
  `Legacy 'M' codes` column names it). R* codes additionally carry `specialist-test-group`
  and `test-method` straight off the Rare & Inherited Disease spreadsheet's own columns;
  current M* codes carry `test-method` from the Cancer directory's "Technology" column
  where that cell holds a clean single value (all but two, as of v16). Verified against
  a hand-regenerated copy already in `FHIRGenomics`'s working tree - this notebook's
  own output now matches it byte-for-byte, so a future rerun won't silently drop these
  properties again.
- `GenomicClinicalIndication` completed the same way from the same two files' 1st-level
  columns (378 R* + 114 M*-current + 103 legacy M*-CI, 595 total) - it was a 16-code
  hand-picked fragment before, one of them (`M4`) with no display text at all. Also
  purely additive once fixed: 2 codes (`R362`/`R94`, both retired "Not present in 8.0"
  tombstones) were nearly dropped by an early version of this regeneration since
  neither master file carries them any more - caught and preserved, the same as
  `EnglandTestCode.fsh`'s own legacy block.
- `DigitalGenomicTestServices` grew from a 2-code placeholder to the real content of
  the two directories that have actually gone digital so far - ~21 `TP` codes, ~350
  `GT` codes - keeping the one pre-existing example pair that belongs to neither file.
  Each `GT` now carries its `TP`(s) as a `parent` `property`
  (`http://hl7.org/fhir/concept-properties#parent`, FHIR's own documented pattern for a
  non-strict hierarchy) rather than nested `concept`s, since 47 `GT` codes answer to
  more than one `TP` - a strict tree couldn't represent that. It also already carries a
  `category` property (added outside this notebook) that this step doesn't yet
  regenerate - see the caveat below.
- Three `ConceptMap`s in total. Two came straight out of data NHS England already
  publishes (the `Legacy 'M' codes` column) - `GenomicTestCode`→`DigitalGenomicTestServices`
  and `GenomicClinicalIndication`→`DigitalGenomicTestServices` - and many-to-many turned
  out to be the norm, not the exception, for the clinical-indication one specifically
  (136 of 175 old CI numbers, 78%, feed more than one new `TP`, since the old haem-onc
  CI grouping was organised around a shared assay, not disease; the test-level map is
  cleaner, 70 of 1304, 5%). The third, `GenomicClinicalIndication`→`GenomicTestCode`,
  is structural rather than empirical (`<CI>.<n>` always belongs under `<CI>`) - fixed
  `equivalence = #narrower` throughout, not the equivalent/relatedto split the other
  two use.
- A handful of codes NHS England's own spreadsheets reference as "legacy" don't
  actually appear in the old `EnglandTestCode.fsh` at all (11 found, two of them oddly
  `R*`-prefixed in a cancer-CNS row) - flagged in Step 3's output rather than silently
  dropped or guessed at.
- **Known gap, not yet addressed here**: Step 2's `DigitalGenomicTestServices.fsh`
  regeneration only ever re-derives `code`/`display`/`parent` from the master
  spreadsheets and the previously-parsed `(code, display)` pairs - `parse_fsh_codes()`
  doesn't read `^property` blocks at all. Its `category` property (and any
  `test-method`/`specialist-test-group` added the same out-of-band way in future) would
  be silently wiped out the next time this notebook actually rewrites that file, since
  nothing here re-populates or preserves them. Out of scope for this pass - flagged so
  it isn't mistaken for something already handled the way `EnglandTestCode.fsh`'s three
  properties now are.
- Nothing here was committed in `FHIRGenomics` - the working tree there now has the
  refreshed files for review before that decision gets made separately.